# Lab 3 - Assignment: Clean the NYC Taxi Data, Then Stop
**Course**: INF4058 Data Engineering  
**Due**: 19th of October 11:59 PM  
**Deliverables**: One PDF Report + GitHub Repo Link  

## Goal
In class we ran the pipeline end-to-end (PostgreSQL in Docker, `ingest_data.py` loading 1,369,765 rows).  
Here we inspect the January 2021 NYC Yellow Taxi dataset, profile all 1,369,765 rows in chunks, identify data quality problems, define cleaning rules, and implement a robust `clean(df)` function.

--- 
## Step 1: Look at the First 100 Rows
- **What is a row?** A single yellow taxi trip record in NYC containing pickup/dropoff timestamps, location IDs, passenger count, distance, fare components, and surcharges.
- **Which columns are dates?** `tpep_pickup_datetime` and `tpep_dropoff_datetime`.
- **Which look empty?** In the first 100 rows, all columns have non-null values. However, full profiling reveals 98,352 rows (~7.18%) have missing metadata in `VendorID`, `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, and `payment_type`.

In [1]:
import pandas as pd
import numpy as np

FILE_PATH = 'yellow_tripdata_2021-01.csv.gz'

# Read first 100 rows
df_100 = pd.read_csv(FILE_PATH, nrows=100)

print('=== df.head() ===')
print(df_100.head())

print('\n=== df.dtypes ===')
print(df_100.dtypes)

print('\n=== Missing Values in First 100 Rows ===')
print(df_100.isnull().sum())

=== df.head() ===
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount

--- 
## Step 2: Profile the Entire Dataset in Chunks
- Read the file in chunks of 100,000 rows without loading all data at once into memory.
- Compute `dtype`, `null_count`, `null_pct`, `min_val`, and `max_val` per column.

In [2]:
chunksize = 100000
total_rows = 0
dtypes_dict = {}
null_counts = {}
min_vals = {}
max_vals = {}

for chunk_idx, chunk in enumerate(pd.read_csv(FILE_PATH, chunksize=chunksize, parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime'], low_memory=False)):
    total_rows += len(chunk)
    if chunk_idx == 0:
        for col in chunk.columns:
            dtypes_dict[col] = str(chunk[col].dtype)
            null_counts[col] = 0
            min_vals[col] = None
            max_vals[col] = None
            
    for col in chunk.columns:
        null_counts[col] += int(chunk[col].isnull().sum())
        valid = chunk[col].dropna()
        if len(valid) > 0:
            c_min = valid.min()
            c_max = valid.max()
            if min_vals[col] is None or c_min < min_vals[col]:
                min_vals[col] = c_min
            if max_vals[col] is None or c_max > max_vals[col]:
                max_vals[col] = c_max

profile_records = []
for col in dtypes_dict.keys():
    cnt = null_counts[col]
    pct = (cnt / total_rows) * 100
    profile_records.append({
        'column': col,
        'dtype': dtypes_dict[col],
        'null_count': cnt,
        'null_pct': f'{pct:.2f}%',
        'min_val': str(min_vals[col]),
        'max_val': str(max_vals[col])
    })

profile_df = pd.DataFrame(profile_records)
print(f'Total Rows Profiled: {total_rows:,}')
print(profile_df.to_string(index=False))

Total Rows Profiled: 1,369,765
               column          dtype  null_count null_pct             min_val             max_val
             VendorID          int64       98352    7.18%                   1                   2
 tpep_pickup_datetime datetime64[us]           0    0.00% 2008-12-31 23:05:14 2021-02-22 16:52:16
tpep_dropoff_datetime datetime64[us]           0    0.00% 2008-12-31 23:07:22 2021-02-22 16:56:15
      passenger_count          int64       98352    7.18%                   0                   8
        trip_distance        float64           0    0.00%                 0.0           263163.28
           RatecodeID          int64       98352    7.18%                   1                  99
   store_and_fwd_flag            str       98352    7.18%                   N                   Y
         PULocationID          int64           0    0.00%                   1                 265
         DOLocationID          int64           0    0.00%                   1          

--- 
## Step 3: Find Problems in the Data
Scan through the dataset in chunks to isolate anomalies:
1. **Pickups outside January 2021**
2. **Negative or zero fares** (`fare_amount <= 0`)
3. **Negative or zero trip distance** (`trip_distance <= 0`)
4. **Passenger count == 0** and **Passenger count is NULL**
5. **Duplicate rows**

In [3]:
outside_jan = 0
fare_bad = 0
dist_bad = 0
both_bad = 0
p_zero = 0
p_null = 0
exact_dups = 0
seen_rows = set()

for chunk in pd.read_csv(FILE_PATH, chunksize=chunksize, parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime'], low_memory=False):
    bad_dates = (chunk['tpep_pickup_datetime'] < '2021-01-01') | (chunk['tpep_pickup_datetime'] >= '2021-02-01')
    outside_jan += int(bad_dates.sum())
    
    f_neg = chunk['fare_amount'] <= 0
    fare_bad += int(f_neg.sum())
    
    d_neg = chunk['trip_distance'] <= 0
    dist_bad += int(d_neg.sum())
    
    both = f_neg & d_neg
    both_bad += int(both.sum())
    
    p_z = chunk['passenger_count'] == 0
    p_zero += int(p_z.sum())
    p_null += int(chunk['passenger_count'].isnull().sum())
    
    for row in chunk.itertuples(index=False, name=None):
        if row in seen_rows:
            exact_dups += 1
        else:
            seen_rows.add(row)

problems_df = pd.DataFrame([
    {'Problem': 'Pickups outside January 2021', 'Count': outside_jan, 'Pct_of_Total': f'{(outside_jan/total_rows)*100:.3f}%'},
    {'Problem': 'Negative or zero fare (fare_amount <= 0)', 'Count': fare_bad, 'Pct_of_Total': f'{(fare_bad/total_rows)*100:.3f}%'},
    {'Problem': 'Negative or zero distance (trip_distance <= 0)', 'Count': dist_bad, 'Pct_of_Total': f'{(dist_bad/total_rows)*100:.3f}%'},
    {'Problem': 'Both fare <= 0 and trip distance <= 0', 'Count': both_bad, 'Pct_of_Total': f'{(both_bad/total_rows)*100:.3f}%'},
    {'Problem': 'Passenger count == 0', 'Count': p_zero, 'Pct_of_Total': f'{(p_zero/total_rows)*100:.3f}%'},
    {'Problem': 'Passenger count is NULL (missing)', 'Count': p_null, 'Pct_of_Total': f'{(p_null/total_rows)*100:.3f}%'},
    {'Problem': 'Exact duplicate rows', 'Count': exact_dups, 'Pct_of_Total': f'{(exact_dups/total_rows)*100:.5f}%'}
])

print('=== Data Problems Found ===')
print(problems_df.to_string(index=False))

=== Data Problems Found ===
                                       Problem  Count Pct_of_Total
                  Pickups outside January 2021     24       0.002%
      Negative or zero fare (fare_amount <= 0)   7411       0.541%
Negative or zero distance (trip_distance <= 0)  19952       1.457%
         Both fare <= 0 and trip distance <= 0    984       0.072%
                          Passenger count == 0  26726       1.951%
             Passenger count is NULL (missing)  98352       7.180%
                          Exact duplicate rows      0     0.00000%


--- 
## Step 4: Decide - Cleaning Decision Matrix
For each problem, we choose **drop**, **fix**, or **flag**, document the rationale, and assign domain ownership.

In [4]:
decision_df = pd.DataFrame([
    {
        'Problem': 'Pickups outside Jan 2021',
        'Decision': 'Drop / Route',
        'Why': 'Belongs to different month/year; corrupts monthly partition.',
        'Owner': 'Data Engineering'
    },
    {
        'Problem': 'Negative or zero fare',
        'Decision': 'Drop',
        'Why': 'Negative fares indicate voids/disputes/cancellations; zero fares indicate unmetered trips.',
        'Owner': 'Finance / Revenue'
    },
    {
        'Problem': 'Negative or zero distance',
        'Decision': 'Drop',
        'Why': 'A taxi trip cannot be 0 miles; indicates meter started mistakenly or GPS glitch.',
        'Owner': 'Operations / Telemetry'
    },
    {
        'Problem': 'Passenger count == 0',
        'Decision': 'Fix (Impute to 1)',
        'Why': 'A completed trip has at least 1 passenger; driver omitted entry; TLC default is 1.',
        'Owner': 'Driver Mobile App'
    },
    {
        'Problem': 'Passenger count is NULL',
        'Decision': 'Fix (Impute to 1)',
        'Why': 'Missing metadata from dispatch records (7.18%); default to 1 passenger.',
        'Owner': 'Data Engineering / Partner'
    },
    {
        'Problem': 'Duplicate rows',
        'Decision': 'Drop',
        'Why': 'Duplicated telemetry logs caused by pipeline network retries.',
        'Owner': 'Data Engineering'
    }
])

print('=== Cleaning Decision Matrix ===')
print(decision_df.to_string(index=False))

=== Cleaning Decision Matrix ===
                  Problem          Decision                                                                                        Why                      Owner
 Pickups outside Jan 2021      Drop / Route                               Belongs to different month/year; corrupts monthly partition.           Data Engineering
    Negative or zero fare              Drop Negative fares indicate voids/disputes/cancellations; zero fares indicate unmetered trips.          Finance / Revenue
Negative or zero distance              Drop           A taxi trip cannot be 0 miles; indicates meter started mistakenly or GPS glitch.     Operations / Telemetry
     Passenger count == 0 Fix (Impute to 1)         A completed trip has at least 1 passenger; driver omitted entry; TLC default is 1.          Driver Mobile App
  Passenger count is NULL Fix (Impute to 1)                    Missing metadata from dispatch records (7.18%); default to 1 passenger. Data Engineering / Par

--- 
## Step 5: Clean - The `clean(df)` Function
Rules followed:
1. **Never drop rows silently** (returns dict of rows removed per rule).
2. **Do not change original dataframe** (operates on `.copy()`).
3. Returns `(cleaned_df, rows_removed_dict)`.

In [5]:
def clean(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    cleaned = df.copy()
    rows_removed = {}
    
    # Ensure datetime types
    cleaned['tpep_pickup_datetime'] = pd.to_datetime(cleaned['tpep_pickup_datetime'])
    cleaned['tpep_dropoff_datetime'] = pd.to_datetime(cleaned['tpep_dropoff_datetime'])
    
    # Rule 1: Drop exact duplicates
    before = len(cleaned)
    cleaned = cleaned.drop_duplicates()
    rows_removed['rule_drop_duplicates'] = before - len(cleaned)
    
    # Rule 2: Drop pickups outside January 2021
    before = len(cleaned)
    valid_dates = (cleaned['tpep_pickup_datetime'] >= '2021-01-01') & (cleaned['tpep_pickup_datetime'] < '2021-02-01')
    cleaned = cleaned[valid_dates]
    rows_removed['rule_drop_outside_jan_2021'] = before - len(cleaned)
    
    # Rule 3: Drop negative or zero fares
    before = len(cleaned)
    valid_fares = cleaned['fare_amount'] > 0
    cleaned = cleaned[valid_fares]
    rows_removed['rule_drop_invalid_fare'] = before - len(cleaned)
    
    # Rule 4: Drop zero or negative trip distance
    before = len(cleaned)
    valid_distance = cleaned['trip_distance'] > 0
    cleaned = cleaned[valid_distance]
    rows_removed['rule_drop_invalid_distance'] = before - len(cleaned)
    
    # Rule 5: Fix / Impute passenger count (0 or null -> 1)
    p_count_0 = int((cleaned['passenger_count'] == 0).sum())
    p_count_null = int(cleaned['passenger_count'].isnull().sum())
    cleaned['passenger_count'] = cleaned['passenger_count'].fillna(1)
    cleaned.loc[cleaned['passenger_count'] == 0, 'passenger_count'] = 1
    rows_removed['rule_fix_passenger_count_0_or_null'] = p_count_0 + p_count_null
    
    return cleaned, rows_removed

# Demonstration on first chunk
demo_chunk = pd.read_csv(FILE_PATH, nrows=100000, low_memory=False)
cleaned_demo, removed_dict = clean(demo_chunk)

print('=== Demonstration on First 100,000 Rows ===')
print('Rows Before:', len(demo_chunk))
print('Rows After: ', len(cleaned_demo))
print('Rows Removed per Rule:', removed_dict)

=== Demonstration on First 100,000 Rows ===
Rows Before: 100000
Rows After:  97889
Rows Removed per Rule: {'rule_drop_duplicates': 0, 'rule_drop_outside_jan_2021': 16, 'rule_drop_invalid_fare': 721, 'rule_drop_invalid_distance': 1374, 'rule_fix_passenger_count_0_or_null': 1923}


--- 
## Step 6: Full Dataset Cleaning Verification (Chunk-by-Chunk)
We apply `clean(df)` iteratively to all chunks of the 1,369,765 rows to calculate total rows before, rows after, and exact drops per rule.

In [6]:
total_raw = 0
total_cleaned = 0
cum_rules = {}

for chunk in pd.read_csv(FILE_PATH, chunksize=chunksize, low_memory=False):
    total_raw += len(chunk)
    cleaned_c, rules = clean(chunk)
    total_cleaned += len(cleaned_c)
    for r, count in rules.items():
        cum_rules[r] = cum_rules.get(r, 0) + count

print('=== FULL DATASET CLEANING SUMMARY (1,369,765 ROWS) ===')
summary_stats = pd.DataFrame([
    {'Metric': 'Total Rows Before Cleaning', 'Value': f'{total_raw:,}'},
    {'Metric': 'Total Rows After Cleaning', 'Value': f'{total_cleaned:,}'},
    {'Metric': 'Total Rows Dropped', 'Value': f'{total_raw - total_cleaned:,}'},
    {'Metric': 'Dataset Retention Rate', 'Value': f'{(total_cleaned / total_raw) * 100:.2f}%'}
])
print(summary_stats.to_string(index=False))

print('\n=== Rows Removed / Handled Per Rule ===')
rules_summary_df = pd.DataFrame([
    {'Rule': r, 'Count': c, 'Pct_of_Raw': f'{(c / total_raw) * 100:.3f}%'}
    for r, c in cum_rules.items()
])
print(rules_summary_df.to_string(index=False))

=== FULL DATASET CLEANING SUMMARY (1,369,765 ROWS) ===
                    Metric     Value
Total Rows Before Cleaning 1,369,765
 Total Rows After Cleaning 1,343,363
        Total Rows Dropped    26,402
    Dataset Retention Rate    98.07%

=== Rows Removed / Handled Per Rule ===
                              Rule  Count Pct_of_Raw
              rule_drop_duplicates      0     0.000%
        rule_drop_outside_jan_2021     24     0.002%
            rule_drop_invalid_fare   7410     0.541%
        rule_drop_invalid_distance  18968     1.385%
rule_fix_passenger_count_0_or_null 123259     8.999%


--- 
## Step 7: Verification
We verify that the cleaned data satisfies all quality constraints:
- 0 rows with pickup outside Jan 2021
- 0 rows with fare <= 0
- 0 rows with trip distance <= 0
- 0 rows with passenger count <= 0 or null
- 0 duplicate rows

In [7]:
# Verify on demonstration cleaned chunk
v_dates = ((cleaned_demo['tpep_pickup_datetime'] < '2021-01-01') | (cleaned_demo['tpep_pickup_datetime'] >= '2021-02-01')).sum()
v_fare = (cleaned_demo['fare_amount'] <= 0).sum()
v_dist = (cleaned_demo['trip_distance'] <= 0).sum()
v_pass = ((cleaned_demo['passenger_count'] <= 0) | cleaned_demo['passenger_count'].isnull()).sum()
v_dup = cleaned_demo.duplicated().sum()

verification_df = pd.DataFrame([
    {'Check': 'Pickups Outside January 2021', 'Invalid Count Remaining': v_dates, 'Status': 'PASSED' if v_dates == 0 else 'FAILED'},
    {'Check': 'Fares <= 0', 'Invalid Count Remaining': v_fare, 'Status': 'PASSED' if v_fare == 0 else 'FAILED'},
    {'Check': 'Trip Distance <= 0', 'Invalid Count Remaining': v_dist, 'Status': 'PASSED' if v_dist == 0 else 'FAILED'},
    {'Check': 'Passenger Count <= 0 or Null', 'Invalid Count Remaining': v_pass, 'Status': 'PASSED' if v_pass == 0 else 'FAILED'},
    {'Check': 'Exact Duplicates', 'Invalid Count Remaining': v_dup, 'Status': 'PASSED' if v_dup == 0 else 'FAILED'},
])
print('=== Cleaned Data Verification ===')
print(verification_df.to_string(index=False))
print('\nAll data quality checks passed! Cleaned data is ready for future pipeline ingestion.')

=== Cleaned Data Verification ===
                       Check  Invalid Count Remaining Status
Pickups Outside January 2021                        0 PASSED
                  Fares <= 0                        0 PASSED
          Trip Distance <= 0                        0 PASSED
Passenger Count <= 0 or Null                        0 PASSED
            Exact Duplicates                        0 PASSED

All data quality checks passed! Cleaned data is ready for future pipeline ingestion.
